In [38]:
import numpy as np

In [82]:
class Neuron:
    def __init__(self, data,children = (),_backward = lambda : None):
        self.data = data
        self.grad = 0
        self.children = children
        self._backward = lambda : None
    def __mul__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        out = Neuron(self.data*other.data)
        out.children = (self,other)
        def _backward():
            self.grad += other.data*out.grad
            other.grad += self.data*out.grad
        out._backward = _backward
        return out
    def __add__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        out = Neuron(self.data + other.data)
        out.children = (self,other)
        def _backward():
            self.grad += 1.0*out.grad
            other.grad += 1.0*out.grad
        out._backward = _backward
        return out
    def __sub__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        out = Neuron(self.data - other.data)
        out.children = (self,other)
        def _backward():
            self.grad += 1.0*out.grad
            other.grad -= 1.0*out.grad
        out._backward = _backward
        return out
    def relu(self):
        out = Neuron(self.data) if self.data > 0 else Neuron(0)
        out.children = (self,)
        def _backward():
            self.grad += out.grad*1 if self.data > 0 else 0
        out._backward = _backward
        return out
    def __truediv__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        def _backward():
            self.grad += out.grad/other.data
            other.grad += -(self.data / (other.data ** 2)) * out.grad
        try:
            out = self.data/other.data
            return Neuron(out , children = (self,other),_backward = _backward)
        except ZeroDivisionError:
            out = Neuron(0)
        def _backward():
            self.grad += out.grad/other.data
            other.grad += -(self.data / (other.data ** 2)) * out.grad
        out.children = (self,other)
        out._backward = _backward
        return out
    def __pow__(self,other : float):
        # other = other if isinstance(other,Neuron) else Neuron(other) we are not doing this cuz
        # if other.data < 0 we might have to face a problem of get undefined
        # to avoid that I am considering other over here as an int variable only
        def _backward():
            self.grad += (other*(self.data**(other - 1)))*out.grad
        out = Neuron(self.data**other)
        out.children = (self,)
        out._backward = _backward
        return out
    def __radd__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        out = Neuron(self.data + other.data)
        out.children = (self,other)
        def _backward():
            self.grad += 1.0*out.grad
            other.grad += 1.0*out.grad
        out._backward = _backward
        return out
    def __rsub__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        out = Neuron(self.data - other.data)
        out.children = (self,other)
        def _backward():
            self.grad += 1.0*out.grad
            other.grad -= 1.0*out.grad
        out._backward = _backward
        return out
    def __rmul__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        out = Neuron(self.data * other.data)
        out.children = (self,other)
        def _backward():
            self.grad += other.data*out.grad
            other.grad += self.data*out.grad
        out._backward = _backward
        return out
    def __rtruediv__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        def _backward():
            self.grad += out.grad/other.data
            other.grad += -(self.data / (other.data ** 2)) * out.grad
        try:
            out = self.data/other.data
            return Neuron(out,children = (self,other),_backward = _backward)
        except ZeroDivisionError:
            out = Neuron(0)
        out.children = (self,other)
        out._backward = _backward
        return out
    def __rpow__(self,other : float):
        out = Neuron(self.data**other)
        def _backward():
            self.grad += (other*(self.data**(other - 1)))*out.grad
        out.children = (self,other)
        out._backward = _backward
        return out
    def __neg__(self):
        out = Neuron(-self.data)
        out.children = (self,)
        def _backward():
            self.grad += -1.0*out.grad
        out._backward = _backward
        return out
    def tanh(self):
        out = Neuron((np.exp(2*self.data) - 1)/(np.exp(2*self.data) + 1))
        out.children = (self,)
        def _backward():
            self.grad += out.grad*(1 - out.data ** 2)
        out._backward = _backward
        return out
    def __repr__(self):
        return f"Neuron({self.data} grad {self.grad})"
    def backward(self):
        self.grad = 1
        visited = set()
        topo = list()
        def topo_sort(n : Neuron,visited : set,topo : list):
            if n not in visited:
                visited.add(n)
            for child in n.children:
                topo_sort(child, visited, topo)
            topo.append(n)
        topo_sort(self,visited, topo)
        for n in reversed(topo):
            n._backward()
class Module:

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0

    def parameters(self):
        return []

class Node(Module):

    def __init__(self, nin, nonlin=True):
        # intializing the default mode be a relu unit with He activation
        self.w = [Neuron(np.random.normal(0,np.sqrt(2/nin))) for _ in range(nin)]
        self.b = Neuron(0)
        self.nin = nin
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return act.relu() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        return f"{'ReLU' if self.nonlin else 'Linear'}Neuron({len(self.w)})"

class Layer(Module):

    def __init__(self, nin, nout,weight_intializer = 'He' ,nonlin = False ,**kwargs):
        self.neurons = [Node(nin, nonlin) for _ in range(nout)]
        def He():
            for neuron in self.neurons:
                neuron.w = [Neuron(np.random.normal(0, np.sqrt(2 / neuron.nin))) for _ in range(neuron.nin)]
        def Xavier():
            for neuron in self.neurons:
                neuron.w = [Neuron(np.random.normal(0, np.sqrt(2 / (neuron.nin + nout)))) for _ in range(neuron.nin)]
        if weight_intializer == 'He':
            He()
        elif weight_intializer == 'Xavier':
            Xavier()
        else:
            raise ValueError(f'Unknown weight initialization : {weight_intializer}')

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def __repr__(self):
        return f"Layer of [{', '.join(str(n) for n in self.neurons)}]"

class MLP(Module):

    def __init__(self, Layers):
        self.x = None
        self.y = None
        self.layers = Layers

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def __repr__(self):
        return f"MLP of [{', '.join(str(layer) for layer in self.layers)}]"

    def fit(self,x,y,optimizer = 'adam',epochs = 1000):
        self.x = x
        self.y = y
        self.optimizer = optimizer
        y_pred = [self(xs) for xs in self.x]
        cost = sum((yout - ygt) ** 2 for ygt, yout in zip(self.y, y_pred))
        self.m = [0]*len(self.parameters())
        self.v = [0]*len(self.parameters())
        self.epoch = 1
        self.epoch_limit = epochs

        def adam(learning_rate = 0.001):
            self.m =[0.9*w1 + 0.1*grad for w1,grad in zip(self.m, [p.grad for p in self.parameters()])]
            self.v =[0.999*v1 + 0.001*(grad**2) for v1,grad in zip(self.v, [p.grad for p in self.parameters()])]
            hat_m = [_m / (1 - 0.9 ** self.epoch) for _m in self.m]
            hat_v = [_v / (1 - 0.999 ** self.epoch) for _v in self.v]
            r_v = [np.sqrt(1/(_hat_v + 0.1**4)) for _hat_v in hat_v]
            for params,_hat_m,_r_v in zip(self.parameters(),hat_m,r_v):
                params.data -= learning_rate*_hat_m*_r_v

        def RMSprop(learning_rate = 0.001):
            self.v =[0.999*v1 + 0.001*(grad**2) for v1,grad in zip(self.v, [p.grad for p in self.parameters()])]
            r_v = [(1/(np.sqrt(_hat_v) + 0.1**4)) for _hat_v in self.v]
            for params,_r_v in zip(self.parameters(),r_v):
                params.data -= learning_rate*_r_v*params.grad

        def momentum(learning_rate = 0.001):
            self.m =[0.9*w1 + 0.1*grad for w1,grad in zip(self.m, [p.grad for p in self.parameters()])]
            for params,_hat_m in zip(self.parameters(),self.m):
                params.data -= learning_rate*_hat_m

        if self.optimizer == 'adam':
            update_step = adam
        elif self.optimizer =='RMSprop':
            update_step = RMSprop
        elif self.optimizer =='momentum':
            update_step = momentum
        else:
            raise ValueError(f'Unknown Optimizer : {self.optimizer}')

        while self.epoch <= self.epoch_limit:
            self.zero_grad()
            y_pred = [self(xs) for xs in self.x]
            cost = sum((yout - ygt) ** 2 for ygt, yout in zip(self.y, y_pred))
            cost.backward()
            update_step()
            if self.epoch%50 == 0:
                print(f"Epoch {self.epoch} | Loss: {cost.data:.4f}")
            self.epoch += 1





In [83]:
x = Neuron(3.0)
a = x * 2 + x * 5 + x**2  # 'a' remembers how to update 'x'
b = a.relu()   # Does this delete 'a's memory?
b.backward()

print(f"x.grad should be 2. It is: {x.grad}")


x.grad should be 2. It is: 13.0


In [85]:

xs = [
    [2.0, 3.0],
    [3.0, -1.0],
    [-0.5, 0.5],
    [1.0, 1.0]
]

ys = [1.0, -1.0, -1.0, 1.0]

n = MLP(
    [
        Layer(3, 2),
        Layer(2, 5),
        Layer(5, 1)
    ]
)
n.fit(xs, ys,optimizer='adam')

print("\nFinal Predictions:")
for x, y in zip(xs, ys):
    pred = n(x)
    print(f"Input: {x} | Target: {y} | Prediction: {pred.data:.4f}")

Epoch 50 | Loss: 120.0695
Epoch 100 | Loss: 74.5347
Epoch 150 | Loss: 50.1524
Epoch 200 | Loss: 35.8467
Epoch 250 | Loss: 26.6429
Epoch 300 | Loss: 20.2423
Epoch 350 | Loss: 15.5663
Epoch 400 | Loss: 12.0736
Epoch 450 | Loss: 9.4565
Epoch 500 | Loss: 7.5139
Epoch 550 | Loss: 6.0964
Epoch 600 | Loss: 5.0818
Epoch 650 | Loss: 4.3671
Epoch 700 | Loss: 3.8664
Epoch 750 | Loss: 3.5115
Epoch 800 | Loss: 3.2520
Epoch 850 | Loss: 3.0526
Epoch 900 | Loss: 2.8906
Epoch 950 | Loss: 2.7521
Epoch 1000 | Loss: 2.6290

Final Predictions:
Input: [2.0, 3.0] | Target: 1.0 | Prediction: 0.9106
Input: [3.0, -1.0] | Target: -1.0 | Prediction: -0.5682
Input: [-0.5, 0.5] | Target: -1.0 | Prediction: 0.4336
Input: [1.0, 1.0] | Target: 1.0 | Prediction: 0.3859
